In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import CompositeSource, HuggingFaceSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

ImportError: cannot import name 'Literal' from 'pydantic' (/home/gpudual/bhklab/declan/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/pydantic/__init__.py)

# Merlin

actual imaging data unavailable right now. 

In [9]:
DATASET_NAME = "Merlin"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



19:19:31 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:82


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver'
]

In [10]:
description = """
This repository provides Merlin Plus, per-voxel annotations for organs and other anatomical structures in the 25,494 CT scans of the Merlin dataset (Stanford). The per-voxel annotations were created by AI models trained on more than 14,000 CT scans at Johns Hopkins University. Merlin Plus is part of a collaboration between the Merlin Project at Stanford and the R-Super Project at Johns Hopkins University. R-Super is a novel AI training methodology that uses radiology reports and organ annotations to significantly improve tumor-segmentation AI. Merlin Plus makes R-Super easily reproducible for the medical AI community.

"""

In [ ]:
source = 

HuggingFaceSource(
    file_type="nifti",
    repo_id="AbdomenAtlas/MerlinPlus",
    description=description,
    post_download=["unzip"]
)

In [12]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [13]:

print(indexed_datasets.source_config(DATASET_NAME))

HuggingFaceSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='huggingface',
    repo_id='AbdomenAtlas/MerlinPlus',
    post_download=['unzip'],
    description='\nThis repository provides Merlin Plus, per-voxel annotations for organs and other anatomical 
structures in the 25,494 CT scans of the Merlin dataset (Stanford). The per-voxel annotations were created by AI 
models trained on more than 14,000 CT scans at Johns Hopkins University. Merlin Plus is part of a collaboration 
between the Merlin Project at Stanford and the R-Super Project at Johns Hopkins University. R-Super is a novel AI 
training methodology that uses radiology reports and organ annotations to significantly improve tumor-segmentation 
AI. Merlin Plus makes R-Super easily reproducible for the medical AI community.\n\n'
)

In [14]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

         [info     ] Downloading all instances from Hugging Face repository AbdomenAtlas/MerlinPlus [imgnet] call=downloaders.download:34
2026-08-12 19:19:34,006 - HTTP Request: GET https://huggingface.co/api/datasets/AbdomenAtlas/MerlinPlus "HTTP/1.1 200 OK"
2026-08-12 19:19:34,050 - HTTP Request: GET https://huggingface.co/api/datasets/AbdomenAtlas/MerlinPlus/revision/main "HTTP/1.1 200 OK"






2026-08-12 19:19:34,144 - HTTP Request: HEAD https://huggingface.co/datasets/AbdomenAtlas/MerlinPlus/resolve/6f4988f5dfbff661092ef066de719e4c3de4a33b/MerlinMasks_BDMAP_00013001_00014000.tar.gz "HTTP/1.1 302 Found"
2026-08-12 19:19:34,156 - HTTP Request: HEAD https://huggingface.co/datasets/AbdomenAtlas/MerlinPlus/resolve/6f4988f5dfbff661092ef066de719e4c3de4a33b/MerlinMasks_BDMAP_00012001_00013000.tar.gz "HTTP/1.1 302 Found"
2026-08-12 19:19:34,157 - HTTP Request: HEAD https://huggingface.co/datasets/AbdomenAtlas/MerlinPlus/resolve/6f4988f5dfbff661092ef066de719e4c3de4a33b/MerlinMasks_BDMAP_00

In [ ]:
crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="images/{SampleID}_{_temp}_{Phase}_{_temp2}.nii.gz",
    mask_name_pattern="labels/{SampleID}_{_temp}_{Phase}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()


In [ ]:
index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

index_df



In [ ]:
index_df

In [ ]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "MR"
    else:
        return "SEG"
    
def map_roi(row):
    if row["file_type"] == "scan":
            return ""
    else:
        return "LESION"
    
index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["ROINames"] = index_df.apply(map_modality, axis=1)

index_df["BodyPartExamined"] = "LIVER"




In [ ]:
import numpy as np


with open(temp_data_path / "LLD_MMRI_Annotation.json", 'r') as f:
    json_data = json.load(f)

annotation_dict = json_data["Annotation_info"]

# Convert your existing dataframe to a dict
df_dict = index_df.to_dict('records')

# Phase mapping for InPhase/OutPhase discrepancy
PHASE_MAPPING = {
    'InPhase': 'In Phase',
    'OutPhase': 'Out Phase',
}

def extract_series_key(filepath, sample_id, phase_from_df):
    """Extract series identifier from filepath."""
    # If phase is InPhase or OutPhase, map it
    if phase_from_df in PHASE_MAPPING:
        return f"{sample_id}_{PHASE_MAPPING[phase_from_df]}"
    return f"{sample_id}_{phase_from_df}"

# Build lookup dictionary
lookup = {}
for sample_id, series_list in annotation_dict.items():
    for series in series_list:
        phase = series.get('phase')
        key = f"{sample_id}_{phase}"
        lookup[key] = series
        
        # Aliases for InPhase/OutPhase
        if phase == 'In Phase':
            lookup[f"{sample_id}_InPhase"] = series
        elif phase == 'Out Phase':
            lookup[f"{sample_id}_OutPhase"] = series

# Update each record
for record in df_dict:
    filepath = record.get('filepath', '')
    if not filepath:
        continue
    
    sample_id = record.get('SampleID', '')
    phase_from_df = record.get('Phase', '')
    
    key = extract_series_key(filepath, sample_id, phase_from_df)
    series_data = lookup.get(key)
    
    if series_data:
        # Basic metadata
        pixel_spacing = series_data.get('pixel_spacing', [0, 0])
        slice_spacing = series_data.get('slice_spacing', 0)
        origin = series_data.get('origin', [0, 0, 0])
        
        record['spacing'] = f"({pixel_spacing[0]}, {pixel_spacing[1]}, {slice_spacing})"
        record['origin'] = f"({origin[0]}, {origin[1]}, {origin[2]})"
        
        # Annotation (lesion) data
        annotation = series_data.get('annotation', {})
        lesion_data = annotation.get('lesion', {})
        
        if '0' in lesion_data:
            bbox = lesion_data['0'].get('bbox', {})
            box_3d = bbox.get('3D_box', {})
            
            if box_3d:
                record['mask.bbox.size'] = box_3d.get('area', 0)
                record['mask.bbox.min_coord'] = f"({box_3d.get('x_min', 0)}, {box_3d.get('y_min', 0)}, {box_3d.get('z_min', 0)})"
                record['mask.bbox.max_coord'] = f"({box_3d.get('x_max', 0)}, {box_3d.get('y_max', 0)}, {box_3d.get('z_max', 0)})"
            
            boxes_2d = bbox.get('2D_box', [])
            if boxes_2d:
                record['num_slices_with_lesion'] = len(boxes_2d)
                record['total_2d_area'] = sum([b.get('area', 0) for b in boxes_2d])

# Convert back to dataframe
df_updated = pd.DataFrame(df_dict)

# Fill ImageID
df_updated['ImageID'] = range(1, len(df_updated) + 1)

# Classify the phase (if you want to keep this)
def classify_phase(phase):
    if phase in ['T2WI']:
        return 'T2_Weighted'
    elif phase in ['DWI']:
        return 'Diffusion_Weighted'
    elif phase in ['C-pre']:
        return 'Pre_Contrast'
    elif phase in ['C+A']:
        return 'Arterial_Phase'
    elif phase in ['C+V']:
        return 'Venous_Phase'
    elif phase in ['C+Delay']:
        return 'Delayed_Phase'
    elif phase in ['InPhase', 'OutPhase']:
        return 'T1_Dual_Echo'
    else:
        return 'Unknown'

df_updated['class'] = df_updated['Phase'].apply(classify_phase)

# Reorder columns
desired_columns = [
    'ImageID', 'ROINames', 'filepath', 'file_type', 'Phase', 'SampleID', 'class',
    'hash', 'size', 'ndim', 'nvoxels', 'spacing', 'origin', 'direction',
    'min', 'max', 'sum', 'mean', 'std', 'variance', 'dtype_str',
    'dtype_numpy', 'mask.bbox.size', 'mask.bbox.min_coord',
    'mask.bbox.max_coord', 'mask.feret_diameter', 'mask.roundness',
    'mask.flatness', 'mask.elongation', 'mask.equivalent_spherical_radius',
    'mask.equivalent_spherical_perimeter', 'mask.equivalent_ellipsoid_diameters',
    'mask.volume_count', 'reference_scan', 'Modality', 'BodyPartExamined'
]

existing_cols = [col for col in desired_columns if col in df_updated.columns]
df_final = df_updated[existing_cols]

print(df_final.head())

In [ ]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet

index_df = df_final
index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)